In [1]:
import pandas as pd
import numpy as np
import torch

def format_stkcd(code):
    return str(code).split('.')[0].zfill(6)

print("按年份切片 (2015-2023)...")

# 1. 加载数据
df_alliance = pd.read_csv("CA_EnterpriseRDAlliance.csv", encoding='utf-8', low_memory=False, on_bad_lines='skip')
df_alliance['symbol'] = df_alliance['symbol'].apply(format_stkcd)
df_alliance = df_alliance.dropna(subset=['inventor'])

df_fin = pd.read_csv("FS_Comins(Merge Query).csv", encoding='utf-8', low_memory=False, on_bad_lines='skip')
df_fin['FS_Comins.Stkcd'] = df_fin['FS_Comins.Stkcd'].apply(format_stkcd)
df_fin = df_fin.drop_duplicates(subset=['FS_Comins.Stkcd'], keep='last')

df_violation = pd.read_csv("STK_Violation_Main.csv", encoding='utf-8', low_memory=False, on_bad_lines='skip')
df_violation['Symbol'] = df_violation['Symbol'].apply(format_stkcd)

# 2. 提取全集
df_alliance['inventor_list'] = df_alliance['inventor'].str.split(';')
df_exploded = df_alliance.explode('inventor_list')
df_exploded['inventor_list'] = df_exploded['inventor_list'].str.strip()

unique_companies = df_exploded['symbol'].unique()
unique_inventors = df_exploded['inventor_list'].unique()

comp_to_idx = {code: i for i, code in enumerate(unique_companies)}
inv_to_idx = {name: i for i, name in enumerate(unique_inventors)}

num_companies = len(unique_companies)

# 生成 9 个动态图切片
years = list(range(2015, 2024))
edge_index_list = [] # 用于存放 9 年的图谱

print("\n切分动态图谱")
for year in years:
    # 筛选出当前年份的联盟数据
    df_year = df_exploded[df_exploded['accper'] == year]
    
    if df_year.empty:
        edge_index_list.append(torch.empty((2, 0), dtype=torch.long))
        continue
        
    src_nodes = df_year['inventor_list'].map(inv_to_idx).values
    dst_edges = df_year['symbol'].map(comp_to_idx).values
    
    edge_index_t = torch.tensor([src_nodes, dst_edges], dtype=torch.long)
    edge_index_list.append(edge_index_t)
    print(f"  - {year}年: 生成技术关联 {edge_index_t.shape[1]} 条")

# 3. 特征 X 和标签 Y 保持不变 (静态属性)
feature_cols = ['FS_Comins.B001101000', 'FS_Comins.B001216000', 'FS_Comins.B002000000', 'FS_Combas.A001000000', 'FS_Combas.A002000000']
X = torch.zeros((num_companies, len(feature_cols)))
for i, comp_code in enumerate(unique_companies):
    row = df_fin[df_fin['FS_Comins.Stkcd'] == comp_code]
    if not row.empty:
        vals = row[feature_cols].fillna(0).values[0]
        X[i] = torch.tensor(vals.astype(float), dtype=torch.float)

violators_set = set(df_violation['Symbol'].unique())
Y = torch.zeros((num_companies, 1))
for i, comp_code in enumerate(unique_companies):
    if comp_code in violators_set:
        Y[i, 0] = 1.0

# 4. 保存时序列表
torch.save(edge_index_list, "dynamic_edge_indices.pt")
torch.save(X, "node_features_X.pt")
torch.save(Y, "risk_labels_Y.pt")

print(f"\n共生成 {len(edge_index_list)} 个年度动态图张量。")

⏳ 正在加载真实数据并按年份切片 (2015-2023)...

🧬 正在按年度切分动态图谱...
  - 2015年: 生成技术关联 85064 条


C:\Users\26841\AppData\Local\Temp\ipykernel_26192\867121938.py:54: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  edge_index_t = torch.tensor([src_nodes, dst_edges], dtype=torch.long)


  - 2016年: 生成技术关联 96045 条
  - 2017年: 生成技术关联 105030 条
  - 2018年: 生成技术关联 120505 条
  - 2019年: 生成技术关联 156764 条
  - 2020年: 生成技术关联 155695 条
  - 2021年: 生成技术关联 178339 条
  - 2022年: 生成技术关联 146177 条
  - 2023年: 生成技术关联 53963 条

✅ 切片完成！共生成 9 个年度动态图张量。
